# M4.12 · Kelime gömmeleri — one-hot, OOV, word2vec

Kategorik bir özniteliği (ör. ocean_proximity) ya da bir kelimeyi ağa vermek: one-hot kodlama ve sözlük dışı (OOV) kovaları, one-hot + biassız yoğun katman = embedding katmanı, "10'dan az kategori one-hot, 50'den fazla embedding" kuralı. Ardından word2vec (Mikolov ve ark. 2013): skip-gram ve CBOW, negatif örneklemeli kayıp ve gradyanları elle, unigram^0.75 gürültü dağılımı ve sık kelime alt-örneklemesi; NumPy'da döngüsüz toplu SGNS ile analojilerin (kral − erkek + kadın ≈ kraliçe) ortaya çıkışı, PyTorch'ta nn.Embedding, seyrek gradyanlar ve CBOW, sonda tam softmax ile negatif örneklemenin maliyet farkı. Ekler: PPMI + SVD (sayım tabanlı gömmeler), Huffman ağaçlı hiyerarşik softmax ve fastText alt-kelime n-gramları.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-12-word2vec.html](https://m-gonul.github.io/pytorch-lab/m4-12-word2vec.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_12_word2vec.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t_vocab():\n        S = [[\"a\", \"b\", \"a\", \"c\"], [\"b\", \"a\", \"d\"]]\n        ensure(need(ns, \"build_vocab\")(S) == {\"a\": 0, \"b\": 1, \"c\": 2, \"d\": 3}, \"sıralama: sayı, sonra alfabe\")\n        ensure(ns[\"build_vocab\"](S, 2) == {\"a\": 0, \"b\": 1}, \"min_count\")\n        v = ns[\"build_vocab\"](S, 2)\n        ensure(need(ns, \"oov_hash\")(\"ab\") == 31 * 97 + 98, \"hash\")\n        e = need(ns, \"encode\")([\"a\", \"zebra\", \"b\", \"zebra\", \"c\"], v, 3)\n        ensure(e[0] == 0 and e[2] == 1 and e[1] == e[3] == 2 + ns[\"oov_hash\"](\"zebra\") % 3 and 2 <= e[4] < 5, \"OOV kovaları\")\n    R_.test(\"sözlük ve OOV\", t_vocab)\n\n    def t_pairs():\n        P = need(ns, \"skipgram_pairs\")([0, 1, 2, 3, 4], 2)\n        ensure(len(P) == 14 and P[:2] == [(0, 1), (0, 2)] and (2, 0) in P and (4, 2) in P and (4, 1) not in P, \"skip-gram çiftleri\")\n        ensure(need(ns, \"cbow_examples\")([5, 6, 7], 1) == [([6], 5), ([5, 7], 6), ([6], 7)], \"CBOW\")\n    R_.test(\"çiftler\", t_pairs)\n\n    def t_grad():\n        J, dv, duo, duk = need(ns, \"sgns_loss_grad\")([1.0], [1.0], [[0.0], [-1.0]])\n        close(J, 1.3196705557466014, what=\"kayıp\")\n        close(dv[0], -0.5378828427399902, what=\"dJ/dv\")\n        rnd = random.Random(0)\n        v = [rnd.gauss(0, 1) for _ in range(4)]\n        uo = [rnd.gauss(0, 1) for _ in range(4)]\n        un = [[rnd.gauss(0, 1) for _ in range(4)] for _ in range(3)]\n        J, dv, duo, duk = ns[\"sgns_loss_grad\"](v, uo, un)\n        h = 1e-6\n        for i in range(4):\n            vp = v[:]\n            vp[i] += h\n            close((ns[\"sgns_loss_grad\"](vp, uo, un)[0] - J) / h, dv[i], atol=1e-4, what=\"sayısal dv\")\n            up = [r[:] for r in un]\n            up[1][i] += h\n            close((ns[\"sgns_loss_grad\"](v, uo, up)[0] - J) / h, duk[1][i], atol=1e-4, what=\"sayısal du_k\")\n        close(need(ns, \"noise_dist\")([81, 16]), [27 / 35, 8 / 35], what=\"3/4 kuvvet\")\n        close(need(ns, \"discard_prob\")(1e-3), 0.9, what=\"alt-örnekleme\")\n        ensure(ns[\"discard_prob\"](1e-6) == 0, \"nadir kelime atılmaz\")\n        rows = []\n        zipf = [1 / r for r in range(1, 1001)]\n        tot = sum(zipf)\n        for pw in (1.0, 0.75, 0.5, 0.0):\n            p = ns[\"noise_dist\"](zipf, pw)\n            rows.append([pw, f\"{p[0]:.3f}\", f\"{sum(p[:10]):.3f}\", f\"{sum(p[500:]):.3f}\"])\n        R_.table(\"Zipf dağılımlı 1000 kelimede gürültü dağılımı\", [\"kuvvet\", \"P(en sık)\", \"P(ilk 10)\", \"P(son 500)\"], rows,\n                 note=\"Kuvvet 1 (ham sıklık) negatif örneklerin çoğunu birkaç işlev kelimesine harcar; 0 (düzgün) nadir kelimelere aşırı ağırlık verir. 3/4 arada bir uzlaşmadır.\")\n    R_.test(\"SGNS kaybı ve gradyanı\", t_grad)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\nTESTS = [(\"man\", \"king\", \"woman\", \"queen\"), (\"man\", \"woman\", \"boy\", \"girl\"), (\"king\", \"queen\", \"prince\", \"princess\"), (\"boy\", \"prince\", \"girl\", \"princess\"),\n         (\"france\", \"paris\", \"spain\", \"madrid\"), (\"italy\", \"rome\", \"japan\", \"tokyo\"), (\"man\", \"boy\", \"queen\", \"princess\"), (\"germany\", \"berlin\", \"egypt\", \"cairo\")]\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"sgns_batch\", \"analogy\", \"most_similar\")])\n    S = ns[\"toy_corpus\"](6000, 0)\n    vocab = {w: i for i, w in enumerate(sorted({w for s in S for w in s}))}\n    V = len(vocab)\n\n    def t_grad():\n        rng = np.random.default_rng(0)\n        Win, Wout = rng.normal(size=(6, 3)), rng.normal(size=(6, 3))\n        c, o, neg = np.array([0, 2, 0]), np.array([1, 3, 1]), np.array([[4, 5], [1, 4], [5, 5]])\n        L, gi, go = need(ns, \"sgns_batch\")(Win, Wout, c, o, neg)\n        h = 1e-6\n        for (M, G) in ((Win, gi), (Wout, go)):\n            for idx in [(0, 1), (2, 0), (5, 2), (1, 1), (3, 2)]:\n                M[idx] += h\n                Lp = ns[\"sgns_batch\"](Win, Wout, c, o, neg)[0]\n                M[idx] -= h\n                close((Lp - L) / h, G[idx], atol=1e-4, what=f\"sayısal gradyan {idx}\")\n        ensure(np.all(gi[[1, 3, 4, 5]] == 0), \"Win'de yalnız merkez satırlar\")\n    R_.test(\"toplu SGNS gradyanı\", t_grad)\n\n    def t_train():\n        P = need(ns, \"make_pairs\")(S, vocab, 3)\n        ensure(P.shape[1] == 2 and len(P) == sum(sum(min(len(s), i + 4) - max(0, i - 3) - 1 for i in range(len(s))) for s in S), \"çift sayısı\")\n        E = need(ns, \"train_sgns\")(P, V, 20, 5, 5, 0.05, 256, 0)\n        res = [(a, b, c, t, need(ns, \"analogy\")(E, vocab, a, b, c)) for a, b, c, t in TESTS]\n        acc = sum(r[3] == r[4] for r in res)\n        ensure(acc >= 7, f\"analoji doğruluğu {acc}/8\")\n        ensure(\"queen\" in need(ns, \"most_similar\")(E, vocab, \"princess\", 3) or \"girl\" in ns[\"most_similar\"](E, vocab, \"princess\", 3), \"komşular\")\n        R_.table(f\"{len(P)} çift, d = 20, 5 epoch — analojiler (b − a + c)\", [\"a : b\", \"c : ?\", \"beklenen\", \"bulunan\"],\n                 [[f\"{a} : {b}\", c, t, f] for a, b, c, t, f in res],\n                 note=\"Derlemde kral−erkek ilişkisini gösteren hiçbir cümle yok: her kelime yalnız özelliklerine (cinsiyet, statü, yaş / ülke–şehir, dil) özgü bağlam kelimeleriyle geçiyor. Skip-gram bu bağlamları tahmin etmeye çalışırken vektörleri özelliklerin toplamı gibi düzenler — analojiler bu toplamsallığın sonucu.\")\n    R_.test(\"eğitim ve analojiler\", t_train)\n",
 "d": "import time\nimport numpy as np\nimport torch\nfrom labcheck import close, need, ensure\n\n\ndef _corpus(n_sent=6000, seed=0):\n    rng = np.random.default_rng(seed)\n    people = {\"man\": (\"m\", \"common\", \"adult\"), \"woman\": (\"f\", \"common\", \"adult\"), \"king\": (\"m\", \"royal\", \"adult\"), \"queen\": (\"f\", \"royal\", \"adult\"),\n              \"boy\": (\"m\", \"common\", \"young\"), \"girl\": (\"f\", \"common\", \"young\"), \"prince\": (\"m\", \"royal\", \"young\"), \"princess\": (\"f\", \"royal\", \"young\")}\n    ctx = {\"m\": [\"he\", \"his\", \"him\"], \"f\": [\"she\", \"her\", \"hers\"], \"common\": [\"street\", \"market\", \"village\"], \"royal\": [\"crown\", \"throne\", \"palace\"],\n           \"adult\": [\"work\", \"tax\", \"old\"], \"young\": [\"school\", \"toy\", \"play\"]}\n    places = [(\"france\", \"paris\", \"french\"), (\"spain\", \"madrid\", \"spanish\"), (\"italy\", \"rome\", \"italian\"),\n              (\"germany\", \"berlin\", \"german\"), (\"japan\", \"tokyo\", \"japanese\"), (\"egypt\", \"cairo\", \"arabic\")]\n    pctx = {\"country\": [\"nation\", \"border\", \"flag\"], \"city\": [\"mayor\", \"downtown\", \"metro\"]}\n    filler = [\"the\", \"a\", \"of\", \"and\", \"is\", \"in\", \"to\", \"with\"]\n    sents = []\n    for _ in range(n_sent):\n        if rng.random() < 0.5:\n            w = list(people)[rng.integers(8)]\n            words = [w] + [str(rng.choice(ctx[a])) for a in people[w]]\n        else:\n            c, city, lang = places[rng.integers(len(places))]\n            w, kind = (c, \"country\") if rng.random() < 0.5 else (city, \"city\")\n            words = [w, lang, str(rng.choice(pctx[kind])), str(rng.choice(pctx[kind]))]\n        words += [str(x) for x in rng.choice(filler, 2)]\n        rng.shuffle(words)\n        sents.append(words)\n    return sents\n\n\nTESTS = [(\"man\", \"king\", \"woman\", \"queen\"), (\"man\", \"woman\", \"boy\", \"girl\"), (\"king\", \"queen\", \"prince\", \"princess\"), (\"boy\", \"prince\", \"girl\", \"princess\"),\n         (\"france\", \"paris\", \"spain\", \"madrid\"), (\"italy\", \"rome\", \"japan\", \"tokyo\"), (\"man\", \"boy\", \"queen\", \"princess\"), (\"germany\", \"berlin\", \"egypt\", \"cairo\")]\n\n\ndef _acc(E, vocab):\n    En = E / E.norm(dim=1, keepdim=True)\n    ok = 0\n    for a, b, c, t in TESTS:\n        s = En @ (En[vocab[b]] - En[vocab[a]] + En[vocab[c]])\n        s[[vocab[a], vocab[b], vocab[c]]] = -1e9\n        ok += int(s.argmax()) == vocab[t]\n    return ok\n\n\ndef check(ns, R_):\n    def t_eq():\n        torch.manual_seed(0)\n        emb = torch.nn.Embedding(7, 3)\n        lin = need(ns, \"embedding_as_linear\")(emb)\n        idx = torch.tensor([2, 5, 2])\n        oh = torch.nn.functional.one_hot(idx, 7).float()\n        close(lin(oh).detach().numpy(), emb(idx).detach().numpy(), atol=1e-6, what=\"one-hot × W = satır seçimi\")\n        (emb(idx) ** 2).sum().backward()\n        (lin(oh) ** 2).sum().backward()\n        close(lin.weight.grad.T.numpy(), emb.weight.grad.numpy(), atol=1e-6, what=\"gradyanlar aynı (yalnız seçilen satırlar)\")\n        m = need(ns, \"SGNS\")(7, 3, True)\n        m(torch.tensor([1, 2]), torch.tensor([3, 4]), torch.tensor([[5, 6], [0, 6]])).backward()\n        g = m.inp.weight.grad\n        ensure(g.is_sparse and set(g.coalesce().indices()[0].tolist()) == {1, 2}, \"sparse=True: yalnız dokunulan satırlar\")\n    R_.test(\"embedding = one-hot × W; seyrek gradyan\", t_eq)\n\n    def t_train():\n        S = _corpus()\n        vocab = {w: i for i, w in enumerate(sorted({w for s in S for w in s}))}\n        V = len(vocab)\n        pad = V\n        sk, cb = [], []\n        for s in S:\n            ids = [vocab[w] for w in s]\n            for i in range(len(ids)):\n                ctx = [ids[j] for j in range(max(0, i - 3), min(len(ids), i + 4)) if j != i]\n                sk += [(ids[i], c) for c in ctx]\n                cb.append((ctx + [pad] * (6 - len(ctx)), ids[i]))\n        sk = torch.tensor(sk)\n        cbc = torch.tensor([c for c, _ in cb])\n        cbo = torch.tensor([o for _, o in cb])\n        cnt = torch.bincount(sk[:, 0], minlength=V).float() ** 0.75\n        g = torch.Generator().manual_seed(0)\n\n        def sk_batches():\n            perm = torch.randperm(len(sk), generator=g)\n            for b in range(0, len(sk), 256):\n                c, o = sk[perm[b:b + 256]].T\n                yield c, o, torch.multinomial(cnt, len(c) * 5, True, generator=g).view(-1, 5)\n\n        def cb_batches():\n            perm = torch.randperm(len(cbo), generator=g)\n            for b in range(0, len(cbo), 64):\n                i = perm[b:b + 64]\n                yield cbc[i], cbo[i], torch.multinomial(cnt, len(i) * 5, True, generator=g).view(-1, 5)\n        rows = []\n        for name, mk, bt, sp in ((\"skip-gram, Adam\", lambda: ns[\"SGNS\"](V, 20), sk_batches, False), (\"skip-gram, seyrek + SparseAdam\", lambda: ns[\"SGNS\"](V, 20, True), sk_batches, True),\n                                 (\"CBOW, Adam\", lambda: need(ns, \"CBOW\")(V, 20, pad), cb_batches, False)):\n            torch.manual_seed(0)\n            m = mk()\n            t0 = time.perf_counter()\n            need(ns, \"train\")(m, bt, 3, 0.01, sp)\n            acc = _acc(m.inp.weight.detach()[:V], vocab)\n            rows.append([name, f\"{acc}/8\", f\"{time.perf_counter() - t0:.1f} sn\"])\n            ensure(acc >= 6, f\"{name}: analojiler {acc}/8\")\n        R_.table(\"Aynı derlem, d = 20, 3 epoch\", [\"model\", \"analoji\", \"süre\"], rows,\n                 note=\"CBOW cümle başına tek örnek üretir (bağlam ortalaması → merkez): daha hızlı, sık kelimelerde iyi; skip-gram her bağlam kelimesi için ayrı örnek üretir: nadir kelimelerde daha iyi (Mikolov). Seyrek gradyan, büyük sözlüklerde her adımda yalnız batch'teki satırları günceller — 10⁶ satırlık tabloda fark büyüktür.\")\n    R_.test(\"SGNS ve CBOW\", t_train)\n",
 "e": "import time\nimport torch\nfrom labcheck import close, need, ensure, fmt_time\n\n\ndef check(ns, R_):\n    def t():\n        torch.manual_seed(0)\n        Win, Wout = torch.randn(50, 8), torch.randn(50, 8)\n        c, o = torch.randint(0, 50, (16,)), torch.randint(0, 50, (16,))\n        s = Win[c] @ Wout.T\n        close(need(ns, \"full_softmax_loss\")(Win, Wout, c, o).item(), (torch.logsumexp(s, 1) - s[torch.arange(16), o]).mean().item(), rtol=1e-5, what=\"tam softmax\")\n        neg = torch.randint(0, 50, (16, 5))\n        ref = -(torch.log(torch.sigmoid((Win[c] * Wout[o]).sum(1))) + torch.log(torch.sigmoid(-(Win[c][:, None] * Wout[neg]).sum(-1))).sum(1)).mean()\n        close(need(ns, \"neg_sampling_loss\")(Win, Wout, c, o, neg).item(), ref.item(), rtol=1e-5, what=\"negatif örnekleme\")\n        rows = []\n        for V in (1000, 10000, 50000):\n            Wi = torch.randn(V, 100, requires_grad=True)\n            Wo = torch.randn(V, 100, requires_grad=True)\n            c, o = torch.randint(0, V, (1024,)), torch.randint(0, V, (1024,))\n            neg = torch.randint(0, V, (1024, 5))\n            r = [V]\n            for f, args in ((ns[\"full_softmax_loss\"], (Wi, Wo, c, o)), (ns[\"neg_sampling_loss\"], (Wi, Wo, c, o, neg))):\n                best = 1e9\n                for _ in range(2):\n                    t0 = time.perf_counter()\n                    f(*args).backward()\n                    best = min(best, time.perf_counter() - t0)\n                r.append(fmt_time(best))\n            rows.append(r)\n        R_.table(\"İleri + geri, batch 1024, d = 100 (CPU)\", [\"V\", \"tam softmax\", \"negatif örnekleme (k = 5)\"], rows,\n                 note=\"Tam softmax batch başına (B × V) logit üretir: V ile doğrusal; negatif örnekleme yalnız B × (k + 1) iç çarpım. Gerçek sözlüklerde (V ~ 10⁶) tam softmax pratik değildir; word2vec'in hızını negatif örnekleme ve hiyerarşik softmax (x2, O(log V)) sağlar. (Burada tam tabloların yoğun gradyanı da süreye katkıda bulunur.)\")\n    R_.test(\"maliyet\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef _corpus(n_sent=6000, seed=0):\n    rng = np.random.default_rng(seed)\n    people = {\"man\": (\"m\", \"common\", \"adult\"), \"woman\": (\"f\", \"common\", \"adult\"), \"king\": (\"m\", \"royal\", \"adult\"), \"queen\": (\"f\", \"royal\", \"adult\"),\n              \"boy\": (\"m\", \"common\", \"young\"), \"girl\": (\"f\", \"common\", \"young\"), \"prince\": (\"m\", \"royal\", \"young\"), \"princess\": (\"f\", \"royal\", \"young\")}\n    ctx = {\"m\": [\"he\", \"his\", \"him\"], \"f\": [\"she\", \"her\", \"hers\"], \"common\": [\"street\", \"market\", \"village\"], \"royal\": [\"crown\", \"throne\", \"palace\"],\n           \"adult\": [\"work\", \"tax\", \"old\"], \"young\": [\"school\", \"toy\", \"play\"]}\n    places = [(\"france\", \"paris\", \"french\"), (\"spain\", \"madrid\", \"spanish\"), (\"italy\", \"rome\", \"italian\"),\n              (\"germany\", \"berlin\", \"german\"), (\"japan\", \"tokyo\", \"japanese\"), (\"egypt\", \"cairo\", \"arabic\")]\n    pctx = {\"country\": [\"nation\", \"border\", \"flag\"], \"city\": [\"mayor\", \"downtown\", \"metro\"]}\n    filler = [\"the\", \"a\", \"of\", \"and\", \"is\", \"in\", \"to\", \"with\"]\n    sents = []\n    for _ in range(n_sent):\n        if rng.random() < 0.5:\n            w = list(people)[rng.integers(8)]\n            words = [w] + [str(rng.choice(ctx[a])) for a in people[w]]\n        else:\n            c, city, lang = places[rng.integers(len(places))]\n            w, kind = (c, \"country\") if rng.random() < 0.5 else (city, \"city\")\n            words = [w, lang, str(rng.choice(pctx[kind])), str(rng.choice(pctx[kind]))]\n        words += [str(x) for x in rng.choice(filler, 2)]\n        rng.shuffle(words)\n        sents.append(words)\n    return sents\n\n\nTESTS = [(\"man\", \"king\", \"woman\", \"queen\"), (\"man\", \"woman\", \"boy\", \"girl\"), (\"king\", \"queen\", \"prince\", \"princess\"), (\"boy\", \"prince\", \"girl\", \"princess\"),\n         (\"france\", \"paris\", \"spain\", \"madrid\"), (\"italy\", \"rome\", \"japan\", \"tokyo\"), (\"man\", \"boy\", \"queen\", \"princess\"), (\"germany\", \"berlin\", \"egypt\", \"cairo\")]\n\n\ndef _acc(E, vocab):\n    En = E / np.linalg.norm(E, axis=1, keepdims=True)\n    ok = 0\n    for a, b, c, t in TESTS:\n        s = En @ (En[vocab[b]] - En[vocab[a]] + En[vocab[c]])\n        s[[vocab[a], vocab[b], vocab[c]]] = -np.inf\n        ok += int(np.argmax(s)) == vocab[t]\n    return ok\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"cooccurrence\", \"ppmi\", \"svd_embeddings\")])\n\n    def t():\n        C = need(ns, \"cooccurrence\")(np.array([[0, 1], [0, 1], [1, 2], [2, 0]]), 3)\n        ensure(C[0, 1] == 2 and C[1, 2] == 1 and C.sum() == 4, \"sayım\")\n        C2 = np.array([[0., 4, 1], [4, 0, 1], [1, 1, 0]])\n        P = need(ns, \"ppmi\")(C2)\n        close(P[0, 1], max(0, np.log(4 * 12 / (5 * 5))), what=\"PPMI\")\n        ensure(P[0, 0] == 0 and np.isfinite(P).all(), \"sıfır sayım → 0\")\n        close(ns[\"ppmi\"](C2, 2.0)[0, 1], max(0, np.log(4 * 12 / 25) - np.log(2)), what=\"SPPMI (kaydırma log k)\")\n        S = _corpus()\n        vocab = {w: i for i, w in enumerate(sorted({w for s in S for w in s}))}\n        V = len(vocab)\n        pairs = np.array([(vocab[s[i]], vocab[s[j]]) for s in S for i in range(len(s)) for j in range(max(0, i - 3), min(len(s), i + 4)) if j != i])\n        Cc = ns[\"cooccurrence\"](pairs, V)\n        rows = []\n        accs = {}\n        for name, M in ((\"ham sayım\", Cc), (\"log(1 + sayım)\", np.log1p(Cc)), (\"PPMI\", ns[\"ppmi\"](Cc))):\n            for d in (5, 8, 20):\n                E = need(ns, \"svd_embeddings\")(M, d)\n                accs[(name, d)] = _acc(E, vocab)\n                rows.append([name, d, f\"{accs[(name, d)]}/8\"])\n        ensure(accs[(\"PPMI\", 8)] >= 6 and accs[(\"PPMI\", 8)] > accs[(\"ham sayım\", 8)], \"düşük boyutta PPMI-SVD ham sayımdan iyi\")\n        R_.table(\"Sayım tabanlı gömmeler: matris → SVD (eğitim yok)\", [\"matris\", \"d\", \"analoji\"], rows,\n                 note=\"Ham sayımlarda sık işlev kelimeleri (the, of…) ilk tekil vektörleri ele geçirir: az boyutla (d = 5, 8) anlam eksenlerine yer kalmaz; bu küçük derlemde d = 20 her şeyi kurtarır, gerçek sözlüklerde kurtarmaz. PMI 'beklenenden ne kadar sık birlikte' sorusunu sorar ve bu baskınlığı kaldırır. Levy & Goldberg: SGNS örtük olarak PMI − log k'yı çarpanlara ayırır — sayım tabanlı ve tahmin tabanlı yöntemler aynı nesnenin iki yüzüdür (GloVe ikisini birleştirir).\")\n    R_.test(\"PPMI + SVD\", t)\n",
 "x2": "import math\nimport random\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        codes, paths = need(ns, \"huffman\")([5, 1, 1, 2])\n        ensure(len(codes[0]) == 1 and len(codes[1]) == 3 and len(codes[2]) == 3 and len(codes[3]) == 2, \"kod uzunlukları (5, 1, 1, 2 → 1, 3, 3, 2)\")\n        ensure(all(paths[w][0] == paths[0][0] for w in range(4)), \"tüm yollar kökten başlar\")\n        cs = [\"\".join(map(str, codes[w])) for w in range(4)]\n        ensure(not any(a != b and b.startswith(a) for a in cs for b in cs), \"önek-özgür kod\")\n        rnd = random.Random(0)\n        for V in (7, 30):\n            counts = [int(1000 / (r + 1)) + 1 for r in range(V)]\n            codes, paths = ns[\"huffman\"](counts)\n            Wn = [[rnd.gauss(0, 1) for _ in range(5)] for _ in range(V - 1)]\n            v = [rnd.gauss(0, 1) for _ in range(5)]\n            close(sum(need(ns, \"hs_prob\")(v, w, codes, paths, Wn) for w in range(V)), 1.0, atol=1e-9, what=f\"V = {V}: olasılıklar toplamı 1\")\n        rows = []\n        for V in (10, 100, 1000, 10000):\n            counts = [int(1e6 / (r + 1)) + 1 for r in range(V)]\n            codes, _ = ns[\"huffman\"](counts)\n            tot = sum(counts)\n            avg = sum(c * len(codes[w]) for w, c in enumerate(counts)) / tot\n            H = -sum(c / tot * math.log2(c / tot) for c in counts)\n            ensure(H <= avg < H + 1, \"Huffman: H ≤ ort. uzunluk < H + 1\")\n            rows.append([V, f\"{math.log2(V):.1f}\", f\"{avg:.2f}\", f\"{H:.2f}\", max(len(c) for c in codes.values())])\n        R_.table(\"Zipf dağılımında Huffman ağacı\", [\"V\", \"log₂ V\", \"ortalama yol (sıklık ağırlıklı)\", \"entropi\", \"en uzun yol\"], rows,\n                 note=\"Hiyerarşik softmax p(w | v)'yi kökten yaprağa ikili kararların çarpımı olarak yazar: V − 1 iç düğüm vektörü, örnek başına yol uzunluğu kadar iç çarpım. Huffman ağacı sık kelimelere kısa yol verir: ortalama maliyet entropiye yakın, log₂ V'den de küçük.\")\n    R_.test(\"hiyerarşik softmax\", t)\n",
 "x3": "import math\nimport random\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        ensure(need(ns, \"char_ngrams\")(\"where\", 3, 3) == [\"<wh\", \"whe\", \"her\", \"ere\", \"re>\", \"<where>\"], \"fastText örneği (Bojanowski ve ark.)\")\n        ensure(\"<her>\" in ns[\"char_ngrams\"](\"her\", 3, 3) and ns[\"char_ngrams\"](\"her\", 3, 6).count(\"<her>\") == 1, \"kısa kelime: tam dizge tek kez\")\n        rnd = random.Random(0)\n        B = 5000\n        Z = [[rnd.gauss(0, 1) for _ in range(64)] for _ in range(B)]\n        cos = lambda a, b: sum(x * y for x, y in zip(a, b)) / math.sqrt(sum(x * x for x in a) * sum(y * y for y in b))\n        wv = lambda w: need(ns, \"word_vector\")(w, Z, B)\n        pairs = [(\"walking\", \"walked\"), (\"walking\", \"talking\"), (\"unhappiness\", \"happiness\"), (\"walking\", \"table\"), (\"tokyo\", \"kyoto\")]\n        rows = [[a, b, f\"{cos(wv(a), wv(b)):.2f}\", len(set(ns['char_ngrams'](a)) & set(ns['char_ngrams'](b)))] for a, b in pairs]\n        ensure(float(rows[2][2]) > float(rows[3][2]) + 0.4 and float(rows[0][2]) > float(rows[3][2]), \"ortak n-gramlar → benzer vektör\")\n        R_.table(\"Rastgele (eğitilmemiş) n-gram vektörleriyle OOV kelime vektörleri\", [\"kelime 1\", \"kelime 2\", \"kosinüs\", \"ortak n-gram\"], rows,\n                 note=\"Eğitimden önce bile, ortak alt dizgileri olan kelimeler ortak vektörleri topladığı için benzerdir; eğitim bu paylaşımı anlamla hizalar. Hiç görülmemiş bir kelimenin ('unhappiness' derlemde yoksa) vektörü n-gramlarından kurulur — OOV kovalarının 'tüm bilinmeyenler aynı' sorununa çözüm. Türkçe gibi eklemeli dillerde özellikle değerli.\")\n    R_.test(\"alt-kelime vektörleri\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m4_12_word2vec as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Kelimeleri (ya da sembolleri) bilgisayara doğrudan veremeyiz; bir temsil gerekir. **One-hot** kodlamada her kategoriye bir indeks verilir; vektör boyu = sözlük + **OOV kovaları**. Test sırasında görülmemiş kategori/kelime (posta kodu, şehir, ürün, kullanıcı…) çıkabilir: bunlar karma (hash) ile birkaç yer tutucu kovadan birine atanır. One-hot öğrenilmez, büyür ve herkes herkese eşit uzaklıktadır (Öklid karesi 2, iç çarpım 0). Kural: ~10'dan az kategori → one-hot, ~50'den fazla → **embedding**: öğrenilebilir yoğun vektör (rastgele başlar, gradyan inişi benzerleri — "NEAR BAY" ile "NEAR OCEAN" — yaklaştırır). One-hot + aktivasyonsuz, biassız yoğun katman = embedding katmanı (ağırlık matrisi = embedding tablosu). Buna **temsil öğrenme** denir; öğrenilen gömmeler başka görevlerde yeniden kullanılabilir (word2vec, GloVe, fastText).

**word2vec** (Mikolov ve ark. 2013): bir kelimeden yakınındaki kelimeleri tahmin eden sığ bir ağ. **Skip-gram**: merkez kelimeden bağlam kelimelerini; **CBOW**: bağlamdan merkezi. Ortaya çıkan uzay anlamlı eksenlere sahiptir: kral − erkek + kadın ≈ kraliçe, Madrid − İspanya + Fransa ≈ Paris. Tam softmax V sınıf üzerinde normalize ettiği için pahalıdır; **negatif örnekleme** her gerçek çifti $k$ "gürültü" çiftinden ayıran ikili sınıflandırmaya çevirir:
$$J=-\log\sigma(u_o^\top v_c)-\sum_{k=1}^K\log\sigma(-u_k^\top v_c),\quad u_k\sim P_n\propto\text{sayı}^{3/4}.$$
Çok sık kelimeler $1-\sqrt{t/f}$ olasılıkla atılır (alt-örnekleme). Alt-kelime yöntemleri (karakter n-gramları, BPE — M4.5) OOV'u çözer; **bağlamsal** gömmeler (ELMo, BERT — M4.16) aynı kelimeye cümleye göre farklı vektör verir.

## 1 · Elle

<!-- çözüm -->

One-hot: $\|e_i-e_j\|^2=1+1=2$. OOV: 10 000 + 100. Embedding: 20 × 10 = 200 (bias yok).

Çiftler: konum 0 → 2 bağlam, 1 → 3, 2 → 4, 3 → 3, 4 → 2: **14**.

Kayıp: $-\log\sigma(1)-\log\sigma(0)-\log\sigma(1)=2\cdot0.3133+0.6931=1.3197$ (ikinci negatifin skoru −1 → $-\log\sigma(+1)$).

Gradyan: $\partial J/\partial v_c=(\sigma(u_o^\top v_c)-1)u_o+\sum_k\sigma(u_k^\top v_c)u_k=(0.731-1)\cdot1+0.5\cdot0+0.269\cdot(-1)=-0.538$. $\partial J/\partial u_o=(\sigma-1)v_c$, $\partial J/\partial u_k=\sigma(u_k^\top v_c)v_c$.

Gürültü: $81^{3/4}=27$, $16^{3/4}=8$ → 27/35 = 0.771 (ham sayılarla 81/97 = 0.835: düzleşme).

Alt-örnekleme: $1-\sqrt{10^{-5}/10^{-3}}=1-0.1=0.9$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**One-hot, OOV, embedding**

- `d2`: İki farklı one-hot vektör arasındaki Öklid uzaklığının karesi
- `oov`: 10 000 kelimelik sözlük + 100 OOV kovası: one-hot vektör boyu
- `emb`: 20 kategori, 10 boyutlu embedding = one-hot + biassız yoğun katman: parametre

**Skip-gram ve negatif örnekleme**

- `pairs`: 5 kelimelik cümle, pencere 2: (merkez, bağlam) çifti sayısı
- `loss`: $u_o\cdot v_c=1$, iki negatif $u_k\cdot v_c\in\{0,-1\}$: $J=-\log\sigma(u_o^\top v_c)-\sum_k\log\sigma(-u_k^\top v_c)$
- `grad`: 1 boyutlu: $v_c=1$, $u_o=1$, $u_1=0$, $u_2=-1$ → $\partial J/\partial v_c$
- `noise`: İki kelime, sayılar 81 ve 16: $P_n\propto c^{3/4}$ ile ilkinin olasılığı
- `disc`: Alt-örnekleme: frekans $f=10^{-3}$, $t=10^{-5}$ → atılma olasılığı $1-\sqrt{t/f}$

In [ ]:
cevaplar = dict(
    d2=None,
    oov=None,
    emb=None,
    pairs=None,
    loss=None,
    grad=None,
    noise=None,
    disc=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"d2":[2,0],"oov":[10100,0],"emb":[200,0],"pairs":[14,0],"loss":[1.3196705557466013,0.000001],"grad":[-0.5378828427399902,0.000001],"noise":[0.7714285714285715,0.000001],"disc":[0.9,1e-9]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `build_vocab(sentences, min_count=1)` → `{kelime: id}`: sayısı ≥ `min_count` olanlar, azalan sayı, eşitlikte alfabetik sıra.
- `oov_hash(word)` — $h\leftarrow(31h+\text{ord}(ch))\bmod 2^{32}$.
- `encode(tokens, vocab, n_oov)` — bilinmeyen kelime → `len(vocab) + oov_hash(w) % n_oov`.
- `skipgram_pairs(ids, window)` — `(merkez, bağlam)` listesi (merkez sırasıyla, bağlam soldan sağa).
- `cbow_examples(ids, window)` → `(bağlam listesi, merkez)`.
- `sgns_loss_grad(v, u_o, U_neg)` → `(J, dv, du_o, [du_k])`, vektörler liste.
- `noise_dist(counts, power=0.75)`; `discard_prob(f, t=1e-5)` → $\max(0, 1-\sqrt{t/f})$.

In [ ]:
%%writefile stage_b.py
def build_vocab(sentences, min_count=1):
    ...


def oov_hash(word):
    ...


def encode(tokens, vocab, n_oov):
    ...


def skipgram_pairs(ids, window):
    ...


def cbow_examples(ids, window):
    ...


def sgns_loss_grad(v, u_o, U_neg):
    ...


def noise_dist(counts, power=0.75):
    ...


def discard_prob(f, t=1e-5):
    ...


print(skipgram_pairs([0, 1, 2, 3, 4], 2))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy. `toy_corpus(n_sent, seed)` verilir: insanlar (cinsiyet × statü × yaş) ve ülke–başkent–dil üçlüleriyle kurulmuş sentetik cümleler — her kelime, özelliklerine özgü bağlam kelimeleriyle birlikte geçer.

- `make_pairs(sentences, vocab, window)` → `(N, 2)` dizi (döngü serbest).
- `sgns_batch(Win, Wout, c, o, neg)` → `(toplam kayıp, gWin, gWout)`: `c, o (B,)`, `neg (B, k)`; gradyanlar tam şekilli dizilerde `np.add.at` ile birikir. Döngüsüz.
- `train_sgns(pairs, V, d=20, k=5, epochs=5, lr=0.05, batch=256, seed=0, counts=None)` → `Win` (doğrusal azalan lr, gürültü $\propto$ sayı$^{0.75}$).
- `analogy(E, vocab, a, b, c)` — kosinüs; $b-a+c$'ye en yakın, a, b, c hariç. Döngüsüz.
- `most_similar(E, vocab, w, n=3)`.

In [ ]:
%%writefile stage_c.py
import numpy as np


def toy_corpus(n_sent=6000, seed=0):
    rng = np.random.default_rng(seed)
    people = {"man": ("m", "common", "adult"), "woman": ("f", "common", "adult"), "king": ("m", "royal", "adult"), "queen": ("f", "royal", "adult"),
              "boy": ("m", "common", "young"), "girl": ("f", "common", "young"), "prince": ("m", "royal", "young"), "princess": ("f", "royal", "young")}
    ctx = {"m": ["he", "his", "him"], "f": ["she", "her", "hers"], "common": ["street", "market", "village"], "royal": ["crown", "throne", "palace"],
           "adult": ["work", "tax", "old"], "young": ["school", "toy", "play"]}
    places = [("france", "paris", "french"), ("spain", "madrid", "spanish"), ("italy", "rome", "italian"),
              ("germany", "berlin", "german"), ("japan", "tokyo", "japanese"), ("egypt", "cairo", "arabic")]
    pctx = {"country": ["nation", "border", "flag"], "city": ["mayor", "downtown", "metro"]}
    filler = ["the", "a", "of", "and", "is", "in", "to", "with"]
    sents = []
    for _ in range(n_sent):
        if rng.random() < 0.5:
            w = list(people)[rng.integers(8)]
            words = [w] + [str(rng.choice(ctx[a])) for a in people[w]]
        else:
            c, city, lang = places[rng.integers(len(places))]
            w, kind = (c, "country") if rng.random() < 0.5 else (city, "city")
            words = [w, lang, str(rng.choice(pctx[kind])), str(rng.choice(pctx[kind]))]
        words += [str(x) for x in rng.choice(filler, 2)]
        rng.shuffle(words)
        sents.append(words)
    return sents


def make_pairs(sentences, vocab, window):
    ...


def sgns_batch(Win, Wout, c, o, neg):
    ...


def train_sgns(pairs, V, d=20, k=5, epochs=5, lr=0.05, batch=256, seed=0, counts=None):
    ...


def analogy(E, vocab, a, b, c):
    ...


def most_similar(E, vocab, w, n=3):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch.

- `embedding_as_linear(emb)` — aynı hesabı yapan `nn.Linear(V, d, bias=False)`.
- `SGNS(V, d)` — `inp`, `out` `nn.Embedding`'ler (`sparse` parametreli); `forward(c, o, neg)` → ortalama SGNS kaybı (`F.logsigmoid`).
- `CBOW(V, d, pad)` — bağlam `(B, 2w)`, eksikler `pad` indeksiyle doldurulmuş (`padding_idx`); bağlam vektörlerinin ortalaması merkez kelimeyi negatif örneklemeyle tahmin eder.
- `train(model, batches, epochs, lr, sparse=False)` — `SparseAdam` (seyrek) ya da `Adam`.

In [ ]:
%%writefile stage_d.py
import time
import torch
import torch.nn as nn
import torch.nn.functional as F


def embedding_as_linear(emb):
    ...


class SGNS(nn.Module):
    def __init__(self, V, d, sparse=False):
        super().__init__()
        ...

    def forward(self, c, o, neg):
        ...


class CBOW(nn.Module):
    def __init__(self, V, d, pad):
        super().__init__()
        ...

    def forward(self, ctx, o, neg):
        ...


def train(model, batches, epochs, lr, sparse=False):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

- `full_softmax_loss(Win, Wout, c, o)` — `cross_entropy(Win[c] @ Wout.T, o)`.
- `neg_sampling_loss(Win, Wout, c, o, neg)`.
- Ölçüm: V ∈ {1 000, 10 000, 50 000}, d = 100, batch 1024, k = 5: ileri + geri süre.

In [ ]:
%%writefile stage_e.py
import time
import torch
import torch.nn.functional as F


def full_softmax_loss(Win, Wout, c, o):
    ...


def neg_sampling_loss(Win, Wout, c, o, neg):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Sayım tabanlı gömmeler: PPMI + SVD

NumPy, döngüsüz (çift üretimi hariç).

- `cooccurrence(pairs, V)` → `(V, V)` sayım matrisi (`np.add.at`).
- `ppmi(C, shift=1.0)` — $\max(0,\log\frac{C_{wc}N}{C_wC_c}-\log\text{shift})$ (shift = k: SPPMI).
- `svd_embeddings(M, d)` → $U_d\sqrt{\Sigma_d}$.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def cooccurrence(pairs, V):
    ...


def ppmi(C, shift=1.0):
    ...


def svd_embeddings(M, d):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Hiyerarşik softmax ve Huffman ağacı

Saf Python + `heapq`.

- `huffman(counts)` → `(codes, paths)`: her kelime için bit listesi ve kökten yaprağa iç düğüm numaraları (iç düğümler 0..V−2).
- `hs_prob(v, w, codes, paths, Wn)` — $p(w\mid v)=\prod_j\sigma(\pm u_{n_j}^\top v)$: bit 0 için +, 1 için −. `v` ve `Wn` listeler.

In [ ]:
%%writefile extra_x2.py
import heapq
import math


def huffman(counts):
    ...


def hs_prob(v, w, codes, paths, Wn):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · fastText: alt-kelime n-gramlarıyla OOV vektörleri

Saf Python.

- `char_ngrams(word, nmin=3, nmax=6)` — `"<" + word + ">"` üzerinde tüm n-gramlar + tam `"<word>"` dizgisi (fastText).
- `ngram_ids(word, buckets)` — her n-gramı `oov_hash % buckets`'a eşle (hash verilir).
- `word_vector(word, Z, buckets)` — n-gram vektörlerinin ortalaması (`Z` liste listesi).

In [ ]:
%%writefile extra_x3.py
import math


def oov_hash(word):
    h = 0
    for ch in word:
        h = (31 * h + ord(ch)) % 2 ** 32
    return h


def char_ngrams(word, nmin=3, nmax=6):
    ...


def ngram_ids(word, buckets):
    ...


def word_vector(word, Z, buckets):
    ...


print(char_ngrams("where", 3, 3))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Mikolov ve ark. (2013b): word2vec'in tam sürümü

[Mikolov, Sutskever, Chen, Corrado, Dean — Distributed Representations of Words and Phrases and their Compositionality (NeurIPS 2013)](https://arxiv.org/abs/1310.4546)

text8 (Wikipedia'nın ilk 100 MB'ı) üzerinde d aşamasındaki SGNS'i alt-örnekleme (Denk. 5), dinamik pencere ve doğrusal azalan öğrenme oranıyla eğit; makaledeki analoji test setinin (Google analogy) anlamsal ve sözdizimsel bölümlerinde doğruluğu raporla. Ardından ifade (phrase) birleştirmeyi uygula: $\mathrm{score}(w_i,w_j)=(c(w_iw_j)-\delta)/(c(w_i)c(w_j))$ ile "new_york" gibi bigramları tek token yap ve vektör toplamsallığını (Tablo 5) göster.


**Adımlar**

1. Alt-örnekleme olmadan / ile eğitim süresi ve analoji doğruluğu tablosu.

**Kabul ölçütleri**

- Anlamsal analojilerde %30'un üzerinde doğruluk (text8, d = 100).
- Alt-örnekleme eğitimi hızlandırır ve nadir kelimelerin vektörlerini iyileştirir.

## Makale ödevi: Levy & Goldberg (2014): SGNS = kaydırılmış PMI'nin çarpanlara ayrılması

[Levy & Goldberg — Neural Word Embedding as Implicit Matrix Factorization (NeurIPS 2014)](https://papers.nips.cc/paper/2014/hash/feab05aa91085b7a8012516bc3533958-Abstract.html)

Makalenin ana sonucunu deneyle doğrula: yeterli boyutta SGNS'in optimumunda $w\cdot c=\mathrm{PMI}(w,c)-\log k$. c aşamasındaki sentetik derlemde ve küçük bir gerçek derlemde k ∈ {1, 5, 15} ve d ∈ {10, 50, V} için öğrenilmiş $W_{in}W_{out}^\top$ ile kaydırılmış PMI arasındaki korelasyonu ölç; x1'deki SPPMI + SVD gömmeleriyle analoji doğruluğunu karşılaştır (Tablo 2).


**Adımlar**

1. Sayımı sıfır olan çiftler için PMI = −∞: değerlendirmeyi gözlenen çiftlerle sınırla.

**Kabul ölçütleri**

- d büyüdükçe korelasyon 1'e yaklaşır.
- SPPMI-SVD ile SGNS'in analoji doğrulukları aynı mertebede.

## Kaynaklar

- [Géron — Hands-On Machine Learning (3. baskı), Bölüm 13 ve 16](https://github.com/ageron/handson-ml3) — Apache-2.0 (kod)
- [Mikolov ve ark. — Efficient Estimation of Word Representations in Vector Space (2013)](https://arxiv.org/abs/1301.3781) — arXiv
- [Mikolov ve ark. — Distributed Representations of Words and Phrases and their Compositionality (NeurIPS 2013)](https://arxiv.org/abs/1310.4546) — arXiv
- [Levy & Goldberg — Neural Word Embedding as Implicit Matrix Factorization (NeurIPS 2014)](https://papers.nips.cc/paper/2014/hash/feab05aa91085b7a8012516bc3533958-Abstract.html) — ©
- [Pennington, Socher, Manning — GloVe (EMNLP 2014)](https://nlp.stanford.edu/pubs/glove.pdf) — ©
- [Bojanowski ve ark. — Enriching Word Vectors with Subword Information (fastText, TACL 2017)](https://arxiv.org/abs/1607.04606) — arXiv
- [Stanford CS224n — Lecture 1–2, Assignment 2 (word2vec)](https://web.stanford.edu/class/cs224n/) — ©